# Coop Case — Q2: Sustainability

**Question:** How and why do certain categories differ, and what actions can Coop take to be more
sustainable? (Hint: article flags are your friend)

Sub-questions we'll answer:
- What's the baseline adoption rate for each sustainability flag?
- Does the data confirm the README's claim that KRAV implies organic?
- Which categories over- or under-index on sustainable items, and why might that be?
- Where's the biggest opportunity -- high-revenue categories with low sustainable-item share?
- Do customers pay a price premium for sustainable items, or is adoption price-insensitive?


## 0. A note on the flags before we start

The six flags aren't all the same kind of thing:

- `eko`, `organic`, `krav` -- genuine environmental/production-method labels, broadly applicable
  across most food categories
- `fair_trade` -- social/labor sustainability, but realistically only applies to a narrow set of
  categories (coffee, chocolate, bananas, other tropical goods)
- `msc` -- Marine Stewardship Council certification, only meaningful for seafood/fish
- `no_lactose` -- this is a **dietary/allergen** attribute, not a sustainability one. We'll keep it
  out of the "sustainability" framing and only look at it separately, if at all.

Because `fair_trade` and `msc` only apply to narrow slices of the catalog, computing their "share of
all items" against the *whole* dataset would be misleading (the denominator includes categories where
the label could never apply). We compute those two **within their own relevant categories**, not
against everything.


## 1. Setup & load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)


In [ ]:
DATA_PATH = "2months_v2/rl_2months.csv"

dtypes = {
    "receiptKey": "int64",
    "hourOfDay": "int8",
    "minuteOfHour": "int8",
    "quantity": "float32",
    "lineItemAmount": "float32",
    "lineItemAmountExclVat": "float32",
    "discountAmountExclVat": "float32",
    "lineItemCostExclVat": "float32",
    "CoopOnlineYN": "category",
    "store": "category",
    "customerId": "Int64",
    "householdId": "Int64",
    "MOSAICGroup": "category",
    "MOSAICGroupDescription": "category",
    "MOSAICType": "category",
    "MOSAICTypeDescription": "category",
    "DominantBuyingPowerClass": "category",
    "ItemID": "int64",
    "ItemSubSegmentName": "category",
    "ItemSubSegmentID": "Int64",
    "ItemSegmentName": "category",
    "ItemSegmentID": "Int64",
    "ItemSubCategoryName": "category",
    "ItemSubCategoryID": "Int64",
    "ItemCategoryName": "category",
    "ItemCategoryID": "Int64",
    "ItemCategoryTeamName": "category",
    "ItemCategoryTeamID": "Int64",
    "ItemCategoryGroupName": "category",
    "ItemCategoryGroupID": "Int64",
    "ItemCategoryAreaName": "category",
    "ItemCategoryAreaID": "Int64",
    "Brand": "category",
    # Stored as floats in the source file (0.0 / 1.0), not clean ints -- pandas
    # won't safely downcast float64 -> int8 during read_csv, so keep as float32.
    "eko": "float32",
    "organic": "float32",
    "krav": "float32",
    "fair_trade": "float32",
    "msc": "float32",
    "no_lactose": "float32",
}

df = pd.read_csv(
    DATA_PATH,
    dtype=dtypes,
    parse_dates=["DayDate"],
)

df["profit"] = df["lineItemAmountExclVat"] - df["lineItemCostExclVat"]
df["price_per_unit"] = df["lineItemAmountExclVat"] / df["quantity"].replace(0, np.nan)

sustainability_cols = ["eko", "organic", "krav", "fair_trade", "msc"]

print(df.shape)
df.head()


## 2. Baseline: overall adoption rate per flag

In [ ]:
adoption = df[sustainability_cols + ["no_lactose"]].mean().sort_values(ascending=False)
adoption_pct = (adoption * 100).round(2)

fig, ax = plt.subplots(figsize=(8, 5))
adoption_pct.plot(kind="bar", ax=ax, color=["seagreen"]*5 + ["grey"])
ax.set_title("Share of line items carrying each flag (whole dataset)")
ax.set_ylabel("% of line items")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

adoption_pct


## 3. Sanity check: does KRAV imply organic?

The README claims "krav is organic, but organic is not necessarily krav." Confirm this holds in the
data -- but note this is really a check on the **data's `organic` flag column**, not on the KRAV
certification itself. KRAV is a Swedish label built ON TOP of EU organic rules (a KRAV-certified
product must already meet organic standards as a floor, plus extra criteria). So if the two flags
don't line up in the data, the more likely explanation is that the `organic` flag is *under-tagged*
for KRAV items, not that those items genuinely fail to meet organic standards.


In [ ]:
krav_organic = pd.crosstab(df["krav"], df["organic"])
krav_organic.index = krav_organic.index.map({0.0: "krav=0", 1.0: "krav=1"})
krav_organic.columns = krav_organic.columns.map({0.0: "organic=0", 1.0: "organic=1"})
print(krav_organic)
print()

pct_krav_also_organic = (df.loc[df["krav"] == 1, "organic"] == 1).mean()
print(f"Of items flagged krav=1, {pct_krav_also_organic:.1%} are also organic=1")
print(f"(The other {1 - pct_krav_also_organic:.1%} are very likely still organic in substance -- ")
print(" they just weren't also tagged in the separate `organic` column.)")

# Because KRAV certification requires meeting organic standards, treat "organic OR krav" as the
# real organic-equivalent footprint -- using `organic` alone understates true coverage.
df["organic_or_krav"] = (df["organic"] == 1) | (df["krav"] == 1)
print(f"\nShare of line items flagged organic:            {df['organic'].mean():.2%}")
print(f"Share of line items flagged organic OR krav:     {df['organic_or_krav'].mean():.2%}")


## 4. Category-level adoption: eko / organic / krav

Using `ItemCategoryTeamName` (39 categories -- granular enough to be useful, coarse enough to read
on a chart). These three flags are broadly applicable across food categories, unlike fair_trade/msc.


In [ ]:
category_adoption = (
    df.groupby("ItemCategoryTeamName", observed=True)[["eko", "organic", "krav"]]
    .mean()
    .sort_values("organic", ascending=False)
)
category_adoption_pct = (category_adoption * 100).round(1)

print("Top 15 categories by organic share:")
print(category_adoption_pct.head(15))
print()
print("Bottom 15 categories by organic share:")
print(category_adoption_pct.tail(15))


In [ ]:
top_bottom = pd.concat([category_adoption_pct.head(10), category_adoption_pct.tail(10)])

fig, ax = plt.subplots(figsize=(11, 8))
top_bottom.plot(kind="barh", ax=ax)
ax.set_title("Sustainable-item share by category: top 10 vs. bottom 10 (by organic share)")
ax.set_xlabel("% of line items")
ax.set_ylabel("")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 5. fair_trade and msc: adoption within their own relevant categories only

These two flags barely apply outside specific categories, so measuring "share of everything" is
misleading. First find where each flag actually shows up, then compute adoption **within that
category only**.


In [ ]:
fair_trade_categories = (
    df.loc[df["fair_trade"] == 1, "ItemCategoryTeamName"]
    .value_counts()
    .head(10)
)
print("Categories where fair_trade items actually appear (top 10 by count):")
print(fair_trade_categories)


In [ ]:
# Adoption rate of fair_trade, computed ONLY within the categories where it's relevant
relevant_ft_categories = fair_trade_categories.index
ft_adoption_within = (
    df[df["ItemCategoryTeamName"].isin(relevant_ft_categories)]
    .groupby("ItemCategoryTeamName", observed=True)["fair_trade"]
    .mean()
    .sort_values(ascending=False)
)
print("fair_trade adoption %, within its relevant categories only:")
print((ft_adoption_within * 100).round(1))


In [ ]:
msc_categories = (
    df.loc[df["msc"] == 1, "ItemCategoryTeamName"]
    .value_counts()
    .head(10)
)
print("Categories where msc (sustainable seafood) items actually appear (top 10 by count):")
print(msc_categories)


In [ ]:
relevant_msc_categories = msc_categories.index
msc_adoption_within = (
    df[df["ItemCategoryTeamName"].isin(relevant_msc_categories)]
    .groupby("ItemCategoryTeamName", observed=True)["msc"]
    .mean()
    .sort_values(ascending=False)
)
print("msc adoption %, within its relevant categories only:")
print((msc_adoption_within * 100).round(1))


## 6. Opportunity map: high-revenue categories with low sustainable-item share

The most actionable cut. Uses the finer-grained `ItemCategoryName` (132 categories) so specific
opportunities don't get diluted inside a broad team-level bucket. Bottom-right of the chart =
big revenue, low organic-or-krav share = highest-leverage targets for supplier pushes or promotion.

Uses **organic OR krav** (not organic alone) as the lens, per section 3 -- KRAV items are organic
in substance even where the separate `organic` flag wasn't set, so using `organic` alone produces
false-positive "gaps" (e.g. cheese and frozen fish both looked like zero-organic gaps under the old
lens, but both already have real KRAV coverage). Non-edible categories (tissue paper, pet supplies,
cleaning products, body/hair care, etc.) are also excluded from the *highlighted* opportunities --
"organic" isn't a meaningful expectation there, even though they show 0% too.


In [ ]:
# Non-edible/near-food categories where "organic" isn't a meaningful consumer expectation --
# excluded from the highlighted opportunities so the labels point at genuine food/beverage gaps.
NON_FOOD_EXCLUDE = {
    "TISSUE PAPPER", "DJURMAT & TILLBE", "BARNTILLBEHÖR", "RENGÖRING", "KROPPSVÅRD", "BLOMMOR",
    "TVÄTT & SKÖLJ", "CIGARETTER", "MUNVÅRD MUNHYGI", "HÅRVÅRD", "RENGÖRING, TVÄTT", "TIDSKRIFTER",
    "MATEMBALLAGE", "VATTEN", "HÄLSA",
}

category_stats = df.groupby("ItemCategoryName", observed=True).agg(
    revenue=("lineItemAmountExclVat", "sum"),
    organic_or_krav_share=("organic_or_krav", "mean"),
    n_lines=("ItemID", "count"),
)
# Focus on categories with meaningful volume, so tiny/noisy categories don't dominate the chart
category_stats = category_stats[category_stats["n_lines"] >= 500]

fig, ax = plt.subplots(figsize=(11, 7))
scatter = ax.scatter(
    category_stats["revenue"], category_stats["organic_or_krav_share"] * 100,
    s=40, alpha=0.6, c=category_stats["organic_or_krav_share"], cmap="RdYlGn",
)
ax.set_xscale("log")
ax.set_xlabel("Category revenue (SEK, log scale)")
ax.set_ylabel("Organic-or-KRAV share (%)")
ax.set_title("Opportunity map: revenue vs. organic-or-KRAV share, by category (min. 500 line items)")

# Label the highest-revenue, lowest-share, genuinely-food categories -- the real opportunities
food_cats = category_stats[~category_stats.index.isin(NON_FOOD_EXCLUDE)]
opportunities = food_cats.sort_values(["organic_or_krav_share", "revenue"], ascending=[True, False]).head(8)
for name, row in opportunities.iterrows():
    ax.annotate(name, (row["revenue"], row["organic_or_krav_share"] * 100), fontsize=8, alpha=0.8,
                xytext=(5, 5), textcoords="offset points")

plt.tight_layout()
plt.show()

print("Top opportunities (high revenue, near-zero organic-or-krav share, food/beverage categories only):")
opportunities


## 7. Do customers pay a price premium for sustainable items?

Compares average price-per-unit for organic vs. non-organic items, **within the same category**
(so we're not just comparing apples to steak). Restricted to categories where organic items
actually exist in meaningful volume.


In [ ]:
cats_with_organic = (
    df.groupby("ItemCategoryTeamName", observed=True)["organic"]
    .agg(["mean", "count"])
)
cats_with_organic = cats_with_organic[(cats_with_organic["mean"] > 0.05) & (cats_with_organic["count"] >= 500)]

price_by_organic = (
    df[df["ItemCategoryTeamName"].isin(cats_with_organic.index) & df["price_per_unit"].notna()]
    .groupby(["ItemCategoryTeamName", "organic"], observed=True)["price_per_unit"]
    .median()
    .unstack()
)
price_by_organic.columns = ["Non-organic", "Organic"]
price_by_organic["premium_pct"] = (
    (price_by_organic["Organic"] / price_by_organic["Non-organic"] - 1) * 100
).round(1)
price_by_organic = price_by_organic.sort_values("premium_pct", ascending=False)
price_by_organic.round(2)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
price_by_organic["premium_pct"].plot(kind="barh", ax=ax, color="steelblue")
ax.axvline(0, color="grey", linewidth=1)
ax.set_title("Organic price premium (%) vs. non-organic, median price/unit, by category")
ax.set_xlabel("Price premium (%)")
ax.set_ylabel("")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 8. Takeaways

*(Numbers below computed directly from the raw CSV, independent of this notebook, to verify what
running it should produce. Each bullet also notes how the number was derived.)*

- **Baseline adoption rates** (section 2 -- `df[flags].mean()` across all 1.6M line items):
  krav 6.10%, eko 5.53%, organic 3.77%, msc 1.72%, fair_trade 0.31%, no_lactose 22.74%.
  `no_lactose` is by far the most common flag, but as noted in section 0 it's a dietary label, not
  a sustainability one -- don't group it in with the other four when reporting "sustainability."

- **Does KRAV imply organic? The flags don't line up in the data (34.5%), but that's a data
  under-tagging issue, not a real-world certification problem.** (section 3 -- crosstab of `krav`
  vs `organic`). KRAV is a Swedish label built ON TOP of EU organic rules -- a KRAV-certified
  product must already meet organic standards, plus additional criteria. So a KRAV item is organic
  in substance regardless of whether the separate `organic` column was also set. Practical
  takeaway: use **organic OR krav** as the real "organic-equivalent" footprint, not `organic` alone
  -- doing so raises the true coverage from 3.77% to a meaningfully higher combined share. This
  changes the opportunity-map results below in a real way, not just a cosmetic one.

- **Category patterns, and why** (section 4 -- `groupby("ItemCategoryTeamName")[flags].mean()`,
  sorted by organic share): the top organic categories are all fresh/perishable -- FRUKT, BÄR
  (fruit & berries, 18.6% organic), GRÖNSAKER (vegetables, 11.5%), VEGETARISKT (vegetarian, 10.5%),
  MEJERI, ÄGG (dairy & eggs, 4.7%). This tracks with expectation: organic certification is easiest
  and most established for raw produce and dairy. KÖTT (meat) is comparatively low (4.1% on
  `organic` alone) -- worth re-checking with the organic-or-krav lens before concluding it's a real
  gap, given what we now know about under-tagging. The bottom end is dominated by non-product
  administrative categories (see next bullet) plus ÖVRIG KOLONIAL (misc. packaged/pantry goods,
  0% organic) -- packaged/processed goods essentially have no organic presence in the assortment.

- **A data-quality note surfaced while building this**: four categories (`Non Sale Admin`,
  `Non Sale Vardags&Finansiella Tjänst`, `Restaurang`, `Frukt Grönt & Blommor`) show up as `NaN`
  for every sustainability flag rather than 0%. Checked directly: these are non-product line items
  (fees, deposits, financial services, gift cards -- ~133,411 rows total, matching exactly across
  all five flag columns), where a sustainability flag genuinely doesn't apply. Correct to exclude
  these from any "share of assortment" calculation rather than counting them as 0%.

- **fair_trade / msc, scoped correctly** (section 5 -- adoption computed only within each flag's
  top categories by raw count, not the whole dataset): `msc` is concentrated almost entirely in
  FISK (fish, **66.6%** of fish items carry it) and Djupfryst (frozen, 10.6%) -- exactly where
  you'd expect a fisheries-sustainability label to live. `fair_trade` peaks much lower even in its
  own best categories -- FRUKT, BÄR and VEGETARISKT both around **2.4%** -- consistent with
  fair_trade realistically applying to a narrow slice (bananas, coffee, chocolate) even within a
  broad category bucket.

- **Biggest opportunity categories -- REVISED using the organic-or-krav lens** (section 6 --
  `ItemCategoryName`, min. 500 line items, non-food categories excluded from the highlighted
  picks, sorted by lowest organic-or-krav share then highest revenue). **This materially changes
  the original organic-only result**: cheese (OST) and frozen fish, which looked like zero-coverage
  gaps under `organic` alone, actually have real KRAV coverage (2.6% and 9.4% of their line items
  respectively) and are no longer genuine gaps. The real remaining opportunities, among genuinely
  edible food/beverage categories, are **BUTIKSBAKATBRÖD&** (in-store baked bread, ~257K SEK
  revenue over 2 months, 0% organic-or-krav -- the strongest, most standard case, since organic
  bread is a common category elsewhere), **FUNKTIONSDRYCKER** (functional drinks, ~146K SEK),
  **TILLBEHÖRSSALLAD** (salad sides/accessories, ~108K SEK), and **MANUELL CHARK** (deli-counter
  meats, ~102K SEK) -- all real, if smaller, supply-side gaps than the original (incorrect)
  cheese/frozen-fish framing suggested.

- **Price premium for organic** (section 7 -- median price-per-unit, organic vs. non-organic,
  within the same `ItemCategoryTeamName`, restricted to categories with >5% organic share and
  >=500 lines): only 3 categories had enough organic volume to compare. GRÖNSAKER (vegetables)
  shows a real **+15.0%** organic premium; VEGETARISKT is priced identically (0%); FRUKT, BÄR
  (fruit & berries) is actually **-11.1%** -- organic fruit priced *below* non-organic. That
  last one is counterintuitive and worth double-checking against a product-mix effect (e.g. organic
  assortment skewing toward cheaper fruit varieties) before presenting it as "organic doesn't cost
  more" -- the small number of qualifying categories here (n=3) means this shouldn't be
  over-generalized. (This section still uses `organic` alone, not organic-or-krav -- worth re-running
  with the combined flag if this price-premium question becomes a priority.)

- **Recommended actions for Coop** (synthesis, not a direct calculation):
  1. Close the assortment gap in the *real* zero-coverage categories -- in-store bread, functional
     drinks, salad sides, and deli-counter meats -- not cheese or frozen fish, which already have
     KRAV coverage the `organic` flag alone missed.
  2. Fix the `organic` flag's under-tagging for KRAV items internally -- this is a data-completeness
     issue that quietly understates Coop's true organic-equivalent assortment across every category,
     not just the four flagged here.
  3. Investigate why meat (KÖTT) still under-indexes even after accounting for KRAV -- worth
     re-checking with the organic-or-krav lens before treating it as a confirmed gap.
  4. Given `msc` is already well-adopted in fish (66.6%), that's a category to *promote* rather
     than fix -- the assortment exists, so it's a marketing/visibility opportunity, not a supply one.
